In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    repo_dir = "/content/stk-mat2011"
    if not os.path.exists(repo_dir):
        !git clone https://github.com/egil10/stk-mat2011.git {repo_dir}
    os.chdir(os.path.join(repo_dir, "code", "notebooks"))
    import sys
    sys.path.append("../scripts")
    from p_drive import mount_drive, sync_data
    mount_drive()
    sync_data("../data/processed/eurusd_clean_returns_jan2026.parquet", to_drive=False)

data_path = Path("../data/processed/eurusd_clean_returns_jan2026.parquet")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | Data: {data_path.exists()}")


Mounted at /content/drive
Loaded eurusd_clean_returns_jan2026.parquet from Google Drive to Local ✅
Device: cuda | Data: True


In [2]:
df = pd.read_parquet(data_path)
df['returns_lag'] = df['returns'].shift(1)
df = df.dropna()

# Features for HMM: Column 0 = return, Column 1 = lagged return
X = df[['returns', 'returns_lag']].values
print(f"Shape for HMM: {X.shape} (Current, Lagged)")


Shape for HMM: (16435, 2) (Current, Lagged)


In [3]:
from hmmlearn.hmm import GaussianHMM

K = 2
best_ll = -np.inf
best_arhmm = None

print(f"Training {K}-State AR(1)-HMM...")
for seed in range(50): # Multi-start for stability
    # We use a 2D Gaussian HMM to capture the joint relationship of (y_t, y_t-1)
    model = GaussianHMM(n_components=K, covariance_type="full", n_iter=1000, random_state=seed)
    try:
        model.fit(X)
        if model.score(X) > best_ll:
            best_ll = model.score(X)
            best_arhmm = model
    except:
        continue

print("Training Complete.")


Training 2-State AR(1)-HMM...


Training Complete.


In [4]:
states = best_arhmm.predict(X)
df['state'] = states

print("-" * 50)
print("REGIME AR(1) COEFFICIENTS:")
print("-" * 50)

for i in range(K):
    # Extract data for this regime
    regime_data = df[df['state'] == i]
    
    # Calculate correlation (proxy for AR coefficient)
    phi = regime_data['returns'].corr(regime_data['returns_lag'])
    sigma = regime_data['returns'].std()
    
    label = "Trending/Momentum" if phi > 0 else "Mean-Reverting"
    print(f"Regime {i} ({label}):")
    print(f"  > AR(1) Phi: {phi:.4f}")
    print(f"  > Volatility: {sigma:.6f}")
    print(f"  > Data Points: {len(regime_data)}")
    print("-" * 30)


--------------------------------------------------
REGIME AR(1) COEFFICIENTS:
--------------------------------------------------
Regime 0 (Trending/Momentum):
  > AR(1) Phi: 0.2807
  > Volatility: 0.000087
  > Data Points: 16380
------------------------------
Regime 1 (Trending/Momentum):
  > AR(1) Phi: 0.1362
  > Volatility: 0.001320
  > Data Points: 55
------------------------------
